### RAG Pipeline - Data Ingestion to Vector DB pipeline

In [ ]:
import sys

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
#processing all pdfs

def process_all_pdfs(pdf_directory):
    all_docs=[]
    pdf_dir=Path(pdf_directory)

    pdfs=list(pdf_dir.glob("*.pdf"))

    for pdf in pdfs:
        try:
            loader=PyMuPDFLoader(str(pdf))
            documents=loader.load() # a doc is created for every page on the pdf

            for doc in documents:
                doc.metadata['source_file']=pdf.name
                doc.metadata['file_type']='pdf'
            
            all_docs.extend(documents) ## appending the docs to all pdf docs
            
        except Exception as e:
            pass
    
    return all_docs

all_pdf_docs=process_all_pdfs('../data')


In [ ]:
### Splitting the text in the docs into chunks

def split_documents(doc,chunk_size=1000,chunk_overlap=200):
    ### making a object 
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n','\n',' ','']
    )
    split_docs=text_splitter.split_documents(doc)

    print(f"Splitted {len(doc)} documents to {len(split_docs)} chunks")

    return split_docs

In [ ]:
all_pdf_chunks=split_documents(all_pdf_docs)


In [ ]:
##importing necessary libraries for embedding the docs

import numpy as np
from sentence_transformers import SentenceTransformer #this is the embedding model 
import chromadb #this is the vector database
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

### Embeddings

In [ ]:
class EmbeddingManager:
    
    def __init__(self,model_name:str='all-MiniLM-L6-v2'):
        
        self.model_name=model_name
        self.model=None
        self.load_model()

    def load_model(self):
        #Loading the SentenceTransformer Model
        try:
            print(f"Loading Embedding Model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding Dimension:{self.model.get_embedding_dimension()}") #384
        except Exception as e:
            print('Could not load model:',e)
    
    def generate(self,texts : List[str]):
        #generate embeddings for a list of text
        #Args:texts= list of text strings to embed
        #returns numpy array of embeddings with shape 

        if not self.model:
            return ValueError("Model Nod Loaded")
        
        print(f"Generating embeddings for {len(texts)} texts:")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}:")
        return embeddings
    
##initializing embedding manager 
embedding_manager=EmbeddingManager()
embedding_manager


### Vector Store

In [ ]:
class VectorStore:

    def __init__(self,collection_name: str='pdf_documents', persist_directory:str='../data/vector_store'):
        

        ##collection name = name of the chromadb collection
        ##persist directory = the directory to store the database

        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self.initialize_store()
    
    def initialize_store(self):
        ##initializing chromadb databse

      
            #initalizing client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            #initializing collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'description':'PDF docs for builfing RAG'}
            )
        
    

    def add_documents(self,documents : List[Any],embeddings:np.ndarray): #here documents is chunk
        
        #adding docs to db

        if(len(documents)!=len(embeddings)): #this is required as we have to insert all the docs with their respective embeddings
            raise ValueError('Number of docs is not equal to number of embeddings')
        
        #prepare data for chromadb

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,emb) in enumerate(zip(documents,embeddings)):
            
            #generate id for each document in a chunk
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #generate metadata

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #document text

            documents_text.append(doc.page_content)

            #embeddings
            embeddings_list.append(emb.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} to the database.")
            print(f"Total documents in collection:{self.collection.count()}")
        except Exception as e:
            print(f"Error adding docs:{e}")
            raise





In [ ]:
vectorstore=VectorStore()
vectorstore.collection.count()

In [ ]:
### Convert text to embeddings
texts=[doc.page_content for doc in all_pdf_chunks] 

#generate embeddings
embeddings=embedding_manager.generate(texts)


In [ ]:
#all the embeddings are generated
embeddings

In [ ]:
#as all 6000 chunks cannot be inserted at once in the chromadb ...hence inserting it in batches

batch=5000
for i in range(0,len(all_pdf_chunks),batch):
    embed_smaller_batch=embeddings[i:i+batch]
    chunks_smaller_batch=all_pdf_chunks[i:i+batch]
    vectorstore.add_documents(chunks_smaller_batch,embed_smaller_batch)

vectorstore

### RAG Retriver

In [ ]:
class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embeggin_manager:EmbeddingManager):

        #will retrieve queries

        self.vector_store=vector_store
        self.embegging_manager=embedding_manager

    def retrieve(self,query:str,top_k: int=5,score_threshold: float = 0.0)  -> List[Dict[str,Any]]:
        #score_threshold=minimum score for entering into output list

        # returns a list of dictionaries containing retrieved documents and metadata

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")

        #generate embedding for query
        query_embedding=embedding_manager.generate([query])[0]

        #search in vs
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #process results
            retrieved=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i,(doc_id,doc,dis,meta)  in enumerate(zip(ids,documents,distances,metadatas)):
                    #convert distance to similarity score(Chroma DB uses cosine distance)
                    sim_score=1-dis
    
                    if sim_score>=score_threshold:
                        retrieved.append({
                            'id':doc_id,
                            'content':doc,
                            'metadata':meta,
                            'distance':dis,
                            'rank':1+i
                        }
                        )
                print(f"Retrived {len(retrieved)} docs")

            else:
                print("No docs found")
            return retrieved
        except Exception as e:
            print(f"Error:{e}")
            return []
            
        

In [ ]:
rag_retriver=RAGRetriever(vectorstore,embedding_manager)
retrieved_docs=rag_retriver.retrieve('What is AI?')

## Integrating Vector DB pipeline with LLM (Augmentation)

In [ ]:
from langchain_groq import ChatGroq
 
import os 
from dotenv import load_dotenv
load_dotenv() ##opens the .env file and makes it available in the code through os.getenv()


##initializing groq key
groq_api_key=os.getenv('GROQ_API_KEY')

llm=ChatGroq(groq_api_key=groq_api_key,model="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)
#temperature controls the randomness and creativity of model...keep it low for RAG models


#Simple RAG function
def rag(query,retriver,llm,top_k=3):
    results=retriver.retrieve(query,top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if len(context)==0:
        return "I could not find any relevant context in the files proveided. Sorry!\nBut I can explain you something else if u like.."

    ##generating answer using LLM
    prompt = f"""
You are an intelligent AI assistant.

Your task is to answer the user's question using ONLY the information provided in the context.

Instructions:

1. Read the context carefully before answering.

2. If the answer is completely or partially available in the context:
   - Explain it clearly in your own words.
   - Do NOT copy sentences directly from the context unless quoting is necessary.
   - Keep the explanation accurate and faithful to the provided information.
   - Use simple language whenever possible.
   - Start with the main idea or intuition before giving detailed explanations.
   - If helpful, include examples, comparisons, or step-by-step reasoning.

3. If the answer is NOT present in the context, respond exactly with:
   "I don't know based on the provided documents."

4. Never use outside knowledge, assumptions, or hallucinations.

5. If the context contains conflicting information, mention the conflict instead of choosing one side.

6. If the user's question is ambiguous, explain what information is missing.

7. Format the answer using Markdown:
   - Use headings when appropriate.
   - Use bullet points for lists.
   - Use numbered steps for procedures.
   - Use tables if comparing information.
   - Highlight important terms using **bold**.

8. Keep the answer concise unless the user asks for a detailed explanation.

Context:
{context}

User Question:
{query}

Answer:
"""

    response=llm.invoke(prompt)
    return response.content

In [ ]:
##simple IO interface
import textwrap
query=input("What do you want to ask?")
response=rag(query,rag_retriver,llm)
print(textwrap.fill(response,width=60,replace_whitespace=False))